In [1]:
import pandas as pd
import dspy
from typing import Literal
import numpy as np


In [2]:
df = pd.read_csv("/Users/nicolasroever/Dropbox/Promotion/Bargaining/bargaining_experiment_analysis/bld/data/merged_data.csv")


In [3]:
class StrategyAnalyzer(dspy.Signature):
    """
    ---CONTEXT---
    After participants read instructions for a bargaining experiment, they are asked to describe their strategy for a bargaining scenario.

    ---TASK---
    Rate whether the strategy makes sense and signals respondent understanding of the bargaining experiment on the discrete scale given in the output field.
    """

    strategy_question: str = dspy.InputField(
        desc="The strategy question that the participant answered."
    )

    strategy_answer: str = dspy.InputField(
        desc="The strategy answer that the participant gave."
    )

    output: Literal["Full Understanding", "Minor Understanging Issues", "Significant Understanging Issues", "No Understanding"] = dspy.OutputField(

    )

    

In [4]:
class StrategyAnalysis(dspy.Module):
    
    def __init__(self):
        super().__init__()
        self.strategy_analyzer = dspy.Predict(StrategyAnalyzer)

    def forward(self, strategy_question: str, strategy_answer: str):
        output = self.strategy_analyzer(strategy_question=strategy_question, strategy_answer=strategy_answer).output    

        return {'output': output}

In [5]:
df["strategy_question"] = np.where(
    df["participant_role"] == "Buyer",
    "Suppose you are a buyer with a valuation of 20. Describe your strategy for the bargaining game.",
    "Suppose you are a seller with a valuation of 0 and a buyer makes an offer of 10. Describe your strategy for the bargaining game. Write at least 2 sentences."
)

In [6]:
lm = dspy.LM(
    model='ollama/deepseek-r1:7b',
    api_base='http://localhost:11434',  
    api_key=''                  
)

dspy.configure(lm=lm)

strategy_analysis = StrategyAnalysis()

In [7]:
data_test = df[df["round"] == 33]

# Create empty lists to store results
output_list = []

# Process each row in the dataframe
for idx, row in data_test.iterrows():
    try:
        output = strategy_analysis(strategy_question=row['strategy_question'], strategy_answer=row['strategy_answer'])
        output_list.append(output['output'])
    except Exception as e:
        # Log the error and append default values
        print(f"Error processing transcript {idx}: {str(e)}")
        output_list.append("error")
    
    print(f"Transcript {idx} processed ({idx}/{len(data_test)})")

# Add new columns to the dataframe
data_test['main_topic'] = output_list

Transcript 29 processed (29/32)
Transcript 59 processed (59/32)
Transcript 89 processed (89/32)
Transcript 119 processed (119/32)
Transcript 149 processed (149/32)
Transcript 178 processed (178/32)
Transcript 208 processed (208/32)
Transcript 238 processed (238/32)
Transcript 268 processed (268/32)
Transcript 297 processed (297/32)
Transcript 327 processed (327/32)
Transcript 357 processed (357/32)
Transcript 387 processed (387/32)
Transcript 417 processed (417/32)
Transcript 447 processed (447/32)
Transcript 477 processed (477/32)
Transcript 507 processed (507/32)
Transcript 537 processed (537/32)
Transcript 567 processed (567/32)
Transcript 597 processed (597/32)
Transcript 627 processed (627/32)
Transcript 657 processed (657/32)
Transcript 687 processed (687/32)
Transcript 717 processed (717/32)
Transcript 747 processed (747/32)
Transcript 777 processed (777/32)
Transcript 807 processed (807/32)
Transcript 837 processed (837/32)


2025/06/12 14:29:39 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


Transcript 867 processed (867/32)
Transcript 897 processed (897/32)
Transcript 927 processed (927/32)
Transcript 957 processed (957/32)


/var/folders/f_/lhmm20kd6r979skh60tngks80000gn/T/ipykernel_29563/4024142816.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_test['main_topic'] = output_list
